In [ ]:
import os
from tokenizers import Tokenizer, decoders, normalizers, pre_tokenizers, Regex
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from datasets import load_dataset

tokenizer_file = "tokenizer-03.json" # file to save
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-100BT", split="train", streaming=True)

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

In [ ]:
def train_tokenizer(dataset, file):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


    tokenizer = Tokenizer(BPE())

    tokenizer.normalizer = normalizers.NFKC()

    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        # Split the text using the regex
        pre_tokenizers.Split(
            pattern=Regex(GPT4_SPLIT_PATTERN),
            behavior="isolated"
        ),
        # Convert the resulting chunks to bytes
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
    ])

    tokenizer.decoder = decoders.ByteLevel()

    # The Trainer (now uses the ByteLevel alphabet)
    trainer = BpeTrainer(
        vocab_size=32768,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=["<|end_text|>", "<|system|>", "<|user|>", "<|assistant|>", "<|end_turn|>", "[PAD]"],
        )


    # Set the number of threads for the Rust backend
    os.environ['RAYON_NUM_THREADS'] = '2'

    print(f"RAYON_NUM_THREADS is set to: {os.environ.get('RAYON_NUM_THREADS')}")

    def batch_iterator(data_split, batch_size=1000):
        # Only keep the text column to avoid decoding the rest of the columns unnecessarily
        tok_dataset = data_split.select_columns("text")
        for batch in tok_dataset.iter(batch_size):
            yield batch["text"]

    dataset = dataset.shuffle(seed=42, buffer_size=50000)

    # TELL THE TOKENIZER TO ONLY LOOK AT THE FIRST 10 MILLION DOCUMENTS
    tokenizer_training_sample = dataset.take(8000000)
    print("Training tokenizer...")
    tokenizer.train_from_iterator(batch_iterator(tokenizer_training_sample), trainer=trainer)

    # save to file
    tokenizer.save(file)
    print(f"Tokenizer saved to {file}")

train_tokenizer(dataset, tokenizer_file)

RAYON_NUM_THREADS is set to: 2
Training tokenizer...



Tokenizer saved to tokenizer-03.json


In [ ]:
# load
tokenizer = Tokenizer.from_file(tokenizer_file)

In [17]:
#tokenizer.add_special_tokens([])

print(f"PAD ID: {tokenizer.token_to_id('[PAD]')}")
print(f"IM_START ID: {tokenizer.token_to_id('<|im_start|>')}")

PAD ID: 30000
IM_START ID: 30001


In [11]:
# encode
output = tokenizer.encode("\sum_{i=1}", add_special_tokens=False)
print(output.tokens)
print(output.ids)

['\\', 's', 'um', '_{', 'i', '=', '1', '}']
[63, 86, 407, 18841, 76, 32, 20, 96]


In [31]:
newline_token = tokenizer.decode([202])
print(f"The model sees the newline character as the token: '{newline_token}'")

The model sees the newline character as the token: '
'


In [62]:
tokenizer.token_to_id("<|assistant|>")

2

In [57]:
tok = tokenizer.encode("Hey whats up?")
tok.ids

[30001, 1, 202, 11729, 621, 86, 516, 34, 3, 202, 30001, 2, 202]

In [58]:
tokenizer.decode(tok.ids, skip_special_tokens=False)

'<|im_start|><|user|>\nHey whats up?<|im_end|>\n<|im_start|><|assistant|>\n'

In [30]:
tokenizer.encode("\n").ids

[202, 0]

In [35]:
tokenizer.get_vocab_size()

30002